# NovaCart — Silver Customers Transformation

This notebook reads the Bronze customers Delta dataset, cleans and standardizes customer fields, validates customer-specific quality rules, routes invalid records to quarantine, and writes trusted records to the Silver layer.

## 1. Import Libraries

In [0]:
# NovaCart - Silver Customers Transformation
#
# Purpose:
# - Read customers from the Bronze Delta table
# - Clean and standardize customer data
# - Validate customer-specific quality rules
# - Write valid records to Silver
# - Write invalid records to the quarantine container

from pyspark.sql import functions as F

## 2. Define Storage Paths

In [0]:

BRONZE_CUSTOMERS_PATH = (
    "abfss://bronze@stnovacartdev.dfs.core.windows.net/"
    "olist/customers"
)

SILVER_CUSTOMERS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/customers"
)

QUARANTINE_CUSTOMERS_PATH = (
    "abfss://quarantine@stnovacartdev.dfs.core.windows.net/"
    "olist/customers"
)

print(f"Bronze path: {BRONZE_CUSTOMERS_PATH}")
print(f"Silver path: {SILVER_CUSTOMERS_PATH}")
print(f"Quarantine path: {QUARANTINE_CUSTOMERS_PATH}")

## 3. Read Bronze Customers Data

In [0]:
customers_bronze_df = (
    spark.read
    .format("delta")
    .load(BRONZE_CUSTOMERS_PATH)
)

bronze_row_count = customers_bronze_df.count()

print("Bronze customers loaded successfully.")
print(f"Bronze row count: {bronze_row_count}")

customers_bronze_df.printSchema()
display(customers_bronze_df.limit(10))

## 4. Profile Bronze Customer Data

In [0]:
customer_profile_df = customers_bronze_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(F.col("customer_id").isNull().cast("int"))
        .alias("missing_customer_id"),

    F.sum(F.col("customer_unique_id").isNull().cast("int"))
        .alias("missing_customer_unique_id"),

    F.sum(F.col("customer_zip_code_prefix").isNull().cast("int"))
        .alias("missing_zip_code_prefix"),

    F.sum(F.col("customer_city").isNull().cast("int"))
        .alias("missing_customer_city"),

    F.sum(F.col("customer_state").isNull().cast("int"))
        .alias("missing_customer_state"),

    F.countDistinct("customer_id")
        .alias("distinct_customer_ids"),

    F.countDistinct("customer_unique_id")
        .alias("distinct_customer_unique_ids")
)

display(customer_profile_df)

## 5. Check Duplicate Customer IDs

In [0]:
duplicate_customer_ids_df = (
    customers_bronze_df
    .groupBy("customer_id")
    .count()
    .filter(
        F.col("customer_id").isNotNull()
        & (F.col("count") > 1)
    )
)

duplicate_customer_id_count = duplicate_customer_ids_df.count()

print(
    f"Number of customer_id values appearing more than once: "
    f"{duplicate_customer_id_count}"
)

display(duplicate_customer_ids_df.limit(20))

## 6. Check Exact Duplicate Records

In [0]:
business_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]

exact_duplicate_count = (
    bronze_row_count
    - customers_bronze_df.dropDuplicates(business_columns).count()
)

print(f"Exact duplicate customer rows: {exact_duplicate_count}")

## 7. Profile Customer Data Quality

In [0]:
customer_quality_profile_df = customers_bronze_df.agg(
    F.sum(
        (
            F.col("customer_id").isNull()
            | (F.trim(F.col("customer_id")) == "")
        ).cast("int")
    ).alias("invalid_customer_id"),

    F.sum(
        (
            F.col("customer_unique_id").isNull()
            | (F.trim(F.col("customer_unique_id")) == "")
        ).cast("int")
    ).alias("invalid_customer_unique_id"),

    F.sum(
        (
            F.col("customer_city").isNull()
            | (F.trim(F.col("customer_city")) == "")
        ).cast("int")
    ).alias("invalid_customer_city"),

    F.sum(
        (
            F.col("customer_state").isNull()
            | (F.trim(F.col("customer_state")) == "")
        ).cast("int")
    ).alias("invalid_customer_state"),

    F.sum(
        (
            F.col("customer_state").isNotNull()
            & ~F.trim(F.col("customer_state")).rlike("^[A-Za-z]{2}$")
        ).cast("int")
    ).alias("invalid_state_format"),

    F.sum(
        (
            F.col("customer_zip_code_prefix").isNull()
            | (F.col("customer_zip_code_prefix") < 0)
            | (F.col("customer_zip_code_prefix") > 99999)
        ).cast("int")
    ).alias("invalid_zip_code_prefix")
)

display(customer_quality_profile_df)

## 8. Inspect Customer State Values

In [0]:
display(
    customers_bronze_df
    .select("customer_state")
    .distinct()
    .orderBy("customer_state")
)

## 9. Clean and Standardize Customer Fields

In [0]:
customers_cleaned_df = (
    customers_bronze_df

    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id"))
    )

    .withColumn(
        "customer_unique_id",
        F.trim(F.col("customer_unique_id"))
    )

    .withColumn(
        "customer_city",
        F.lower(F.trim(F.col("customer_city")))
    )

    .withColumn(
        "customer_state",
        F.upper(F.trim(F.col("customer_state")))
    )
)

## 10. Inspect Cleaned Customer Data

In [0]:
display(
    customers_cleaned_df.select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ).limit(20)
)

## 11. Define Customer Validation Rules

In [0]:
invalid_customer_id_condition = (
    F.col("customer_id").isNull()
    | (F.col("customer_id") == "")
)

invalid_customer_unique_id_condition = (
    F.col("customer_unique_id").isNull()
    | (F.col("customer_unique_id") == "")
)

invalid_customer_city_condition = (
    F.col("customer_city").isNull()
    | (F.col("customer_city") == "")
)

invalid_customer_state_condition = (
    F.col("customer_state").isNull()
    | (F.col("customer_state") == "")
    | ~F.col("customer_state").rlike("^[A-Z]{2}$")
)

invalid_zip_code_condition = (
    F.col("customer_zip_code_prefix").isNull()
    | (F.col("customer_zip_code_prefix") < 0)
    | (F.col("customer_zip_code_prefix") > 99999)
)

## 12. Assign Rejection Reasons

In [0]:
customers_validated_df = customers_cleaned_df.withColumn(
    "_rejection_reason",

    F.when(
        invalid_customer_id_condition,
        F.lit("MISSING_CUSTOMER_ID")
    )

    .when(
        invalid_customer_unique_id_condition,
        F.lit("MISSING_CUSTOMER_UNIQUE_ID")
    )

    .when(
        invalid_customer_city_condition,
        F.lit("MISSING_CUSTOMER_CITY")
    )

    .when(
        invalid_customer_state_condition,
        F.lit("INVALID_CUSTOMER_STATE")
    )

    .when(
        invalid_zip_code_condition,
        F.lit("INVALID_ZIP_CODE_PREFIX")
    )

    .otherwise(F.lit(None))
)

## 13. Review Validation Results

In [0]:
display(
    customers_validated_df
    .groupBy("_rejection_reason")
    .count()
    .orderBy("_rejection_reason")
)

## 14. Split Valid and Invalid Records

In [0]:
customers_valid_df = (
    customers_validated_df
    .filter(F.col("_rejection_reason").isNull())
    .drop("_rejection_reason")
)

customers_quarantine_df = (
    customers_validated_df
    .filter(F.col("_rejection_reason").isNotNull())
)

## 15. Add Silver Processing Metadata

In [0]:
customers_silver_df = (
    customers_valid_df
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

## 16. Add Quarantine Metadata

In [0]:
customers_quarantine_df = (
    customers_quarantine_df
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_dataset",
        F.lit("customers")
    )
)

## 17. Count Silver and Quarantine Records

In [0]:
valid_row_count = customers_silver_df.count()
quarantine_row_count = customers_quarantine_df.count()

print(f"Valid Silver rows: {valid_row_count}")
print(f"Quarantined rows: {quarantine_row_count}")
print(f"Bronze input rows: {bronze_row_count}")

## 18. Validate Row-Count Reconciliation

In [0]:
if valid_row_count + quarantine_row_count != bronze_row_count:
    raise ValueError(
        "Row-count validation failed: "
        "Silver rows + quarantine rows do not equal Bronze input rows."
    )

print("Row-count validation passed.")

## 19. Write Valid Customers to Silver

In [0]:
(
    customers_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_CUSTOMERS_PATH)
)

print("Silver customers written successfully.")

## 20. Write Invalid Customers to Quarantine

In [0]:
(
    customers_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(QUARANTINE_CUSTOMERS_PATH)
)

print("Customer quarantine output written successfully.")

## 21. Read Written Delta Outputs

In [0]:
customers_silver_written_df = (
    spark.read
    .format("delta")
    .load(SILVER_CUSTOMERS_PATH)
)

customers_quarantine_written_df = (
    spark.read
    .format("delta")
    .load(QUARANTINE_CUSTOMERS_PATH)
)

silver_written_count = customers_silver_written_df.count()
quarantine_written_count = customers_quarantine_written_df.count()

print(f"Written Silver rows: {silver_written_count}")
print(f"Written quarantine rows: {quarantine_written_count}")

## 22. Validate Written Outputs

In [0]:
if silver_written_count != valid_row_count:
    raise ValueError(
        "Silver write validation failed: "
        f"expected {valid_row_count}, wrote {silver_written_count}."
    )

if quarantine_written_count != quarantine_row_count:
    raise ValueError(
        "Quarantine write validation failed: "
        f"expected {quarantine_row_count}, wrote "
        f"{quarantine_written_count}."
    )

if silver_written_count + quarantine_written_count != bronze_row_count:
    raise ValueError(
        "Final reconciliation failed: "
        "Silver + quarantine does not equal Bronze."
    )

print("Silver customers pipeline completed successfully.")
print("Final row-count validation passed.")

## 23. Inspect Final Silver Customers Dataset

In [0]:
customers_silver_written_df.printSchema()

display(
    customers_silver_written_df.limit(20)
)